# URECA: Random Forest Prediction of Fluorophore Photophysical Properties

**Project:** NTU URECA (self-directed), supervised by Prof Liu

This is a single notebook covering the whole pipeline: setup, EDA, featurization, modeling, analysis, and write-up. Everything is built in order, in this one file.

---

## Goal

We're predicting four photophysical properties of organic fluorophores (chromophores) from molecular structure (SMILES string):

| Target | Column in raw data | Notes |
|---|---|---|
| Absorption max | `Absorption max (nm)` | |
| Emission max | `Emission max (nm)` | |
| Extinction coefficient | `log(e/mol-1 dm3 cm-1)` | already log10-scaled in the raw data |
| Quantum yield (Φ_F) | `Quantum yield` | bounded 0–1, expected to be the hardest target |

**Model:** Random Forest, as the first model.

**Key method:** train/test split by **Murcko scaffold** (via RDKit), not randomly. A random split can let near-identical molecules appear on both sides, so the model can do well just by memorizing similar structures. Comparing performance on a scaffold split vs. a random split shows how well the model actually generalizes to new chemical structures — this gap is a potential result worth reporting.

## Notebook sections

1. **Setup & data loading**
2. EDA — distributions, missingness, solvent effects, correlations
3. Featurization + scaffold split
4. RF tuning
5. Feature importance + scaffold-vs-random gap analysis
6. Error analysis
7. Write-up

## Dataset

`chromophore_data.csv`, in this same folder. 20,836 rows, one row per (chromophore, solvent) measurement. Columns: SMILES, solvent, the four target properties, plus lifetime, FWHM, molecular weight, and reference.

Two things to note: not every row has all four targets (missingness varies by target), and the same molecule can appear multiple times under different solvents. Both are covered in the EDA section below.

## 1. Imports

`pandas`/`numpy` for data handling, `matplotlib`/`seaborn` for plots, `rdkit` for chemistry (reading SMILES, scaffolds, descriptors), `scikit-learn` for the model and evaluation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from rdkit import Chem
from rdkit.Chem import Draw, Descriptors
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')  # RDKit likes to complain about every weird SMILES - silence that for now

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 50)

print('rdkit, sklearn, pandas all imported OK')

## 2. Load the raw dataset

Read the CSV from this folder. `df_raw` stays untouched as a backup; all work happens on `df`.

In [ ]:
DATA_PATH = 'chromophore_data.csv'

df_raw = pd.read_csv(DATA_PATH, low_memory=False)
df = df_raw.copy()  # work on a copy, keep df_raw untouched as a backup

print(f'Loaded {df.shape[0]:,} rows x {df.shape[1]} columns')
df.head()

## 3. Rename columns

The raw column names (e.g. `Absorption max (nm)`) are long to type repeatedly, so we rename them to short codes. `df_raw` keeps the original names.

In [ ]:
RENAME_MAP = {
    'Chromophore': 'smiles',
    'Solvent': 'solvent',
    'Absorption max (nm)': 'abs_max',
    'Emission max (nm)': 'emi_max',
    'Quantum yield': 'qy',
    'log(e/mol-1 dm3 cm-1)': 'log_ext_coeff',
    'Molecular weight (g mol-1)': 'mol_weight',
}

df = df.rename(columns=RENAME_MAP)

# these two lists get reused constantly from here on, so let's name them once
TARGET_COLS = ['abs_max', 'emi_max', 'log_ext_coeff', 'qy']
ID_COLS = ['smiles', 'solvent']

df[ID_COLS + TARGET_COLS].head()

## 4. Sanity checks

Three checks before doing anything else: how much data each target actually has, whether RDKit can parse every SMILES string, and how many unique molecules we have (since the same molecule appears under multiple solvents).

In [ ]:
print('Non-null counts per target (out of {:,} rows):'.format(len(df)))
print(df[TARGET_COLS].notna().sum())
print()
print('Unique SMILES (molecules):', df['smiles'].nunique())
print('Unique solvents:', df['solvent'].nunique())

In [ ]:
def parses_ok(smi):
    # if RDKit can't parse a SMILES, Chem.MolFromSmiles just quietly returns None
    try:
        return Chem.MolFromSmiles(smi) is not None
    except Exception:
        return False

parse_mask = df['smiles'].apply(parses_ok)
print(f'SMILES that RDKit can parse: {parse_mask.sum():,} / {len(df):,} ({parse_mask.mean():.1%})')

if (~parse_mask).any():
    print('\nExample unparseable SMILES:')
    display(df.loc[~parse_mask, ['smiles']].head())

## 5. Exploratory Data Analysis

Go through the four targets one at a time: distribution, missingness, outliers, then a short findings note before moving to the next. After all four, look at cross-target relationships (e.g. Stokes shift = emission − absorption) and solvent effects.

### 5.1 Absorption max (nm)

In [ ]:
# how many rows actually have this value, and what does it look like on average?
print(f"Missing: {df['abs_max'].isna().sum():,} / {len(df):,} rows ({df['abs_max'].isna().mean():.1%})")
df['abs_max'].describe()

In [ ]:
# histogram for the overall shape, boxplot to spot outliers at a glance
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(df['abs_max'].dropna(), bins=60, ax=axes[0], color='steelblue')
axes[0].set_title('Absorption max — distribution')
axes[0].set_xlabel('Absorption max (nm)')

sns.boxplot(x=df['abs_max'].dropna(), ax=axes[1], color='steelblue')
axes[1].set_title('Absorption max — boxplot (outlier check)')
axes[1].set_xlabel('Absorption max (nm)')

plt.tight_layout()
plt.show()

In [ ]:
# real UV-Vis-NIR absorption lives roughly in the 200-1000 nm window,
# so anything outside that is worth eyeballing - could be a typo in the source paper
low = df[df['abs_max'] < 200][['smiles', 'solvent', 'abs_max']]
high = df[df['abs_max'] > 1000][['smiles', 'solvent', 'abs_max']]

print(f'Below 200 nm: {len(low)} rows')
print(f'Above 1000 nm: {len(high)} rows')
display(pd.concat([low, high]))

**Findings — Absorption max:**

- **Coverage:** 17,703 / 20,836 rows (~85%) — the best-covered target.
- **Shape:** 127.7–1055.0 nm, median 397 nm, right-skewed (skew ≈ 1.25). Most values fall in the near-UV/visible range (350–480 nm is the middle 50%), with a long tail toward the NIR.
- **Outliers checked:** 21 rows below 200 nm and 1 above 1000 nm. All are explainable — the sub-200nm rows are small gas-phase molecules (methane, ethane, propane, etc.) that genuinely absorb in the far-UV; the >1000nm row is a large conjugated dye. None are data errors, so nothing gets dropped.
- **No other issues** — no zeros or negatives.
- **For modeling:** Random Forest handles skew and outliers fine, so no transform is needed for this target.

Next: Emission max.

### 5.2 Emission max (nm)

Same checks as above, plus one extra: emission should physically occur at a longer wavelength than absorption (Stokes shift), so we can use that as a sanity check.

In [ ]:
print(f"Missing: {df['emi_max'].isna().sum():,} / {len(df):,} rows ({df['emi_max'].isna().mean():.1%})")
df['emi_max'].describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(df['emi_max'].dropna(), bins=60, ax=axes[0], color='darkorange')
axes[0].set_title('Emission max — distribution')
axes[0].set_xlabel('Emission max (nm)')

sns.boxplot(x=df['emi_max'].dropna(), ax=axes[1], color='darkorange')
axes[1].set_title('Emission max — boxplot (outlier check)')
axes[1].set_xlabel('Emission max (nm)')

plt.tight_layout()
plt.show()

In [ ]:
# physics check: a molecule loses a bit of energy before it emits, so emission
# wavelength should always be >= absorption wavelength (positive Stokes shift).
# if we see the opposite, that's either unusual photophysics or a data issue worth a look.
both = df.dropna(subset=['emi_max', 'abs_max']).copy()
both['stokes_shift'] = both['emi_max'] - both['abs_max']

neg_shift = both[both['stokes_shift'] < 0].sort_values('stokes_shift')
print(f"Rows with emission < absorption (negative Stokes shift): {len(neg_shift)} / {len(both)}")
display(neg_shift[['smiles', 'solvent', 'abs_max', 'emi_max', 'stokes_shift']].head(10))

**Findings — Emission max:**

- **Coverage:** 18,851 / 20,836 rows (~90%) — better covered than absorption.
- **Shape:** 247–1050 nm, median 488 nm, mildly right-skewed (skew ≈ 0.66, less skewed than absorption). Consistent with the Stokes shift pushing emission to longer wavelengths than absorption.
- **Outliers:** only 1 row above 1000 nm (a long conjugated dye, plausible), none below 200 nm.
- **Stokes shift check:** 17 / 16,422 rows (~0.1%) have emission at a shorter wavelength than absorption, which shouldn't normally happen. 15 are small (≤15 nm), likely measurement noise. 2 are larger (-57, -55 nm, both BODIPY-dimer structures in toluene) — worth a second look later, but too few rows to drop.
- **For modeling:** cleaner than absorption max, no transform needed.

Next: log(extinction coefficient).